# SQL Joins & Multi-Table Analysis

Analysis of customer-order relationships, unmatched keys, join types, and a three-table enrichment join.

In [ ]:
import sqlite3
import pandas as pd

customers = pd.read_csv('../data/raw/customers_join_data.csv')
orders = pd.read_csv('../data/raw/orders.csv')
features = pd.read_csv('../data/raw/customer_features.csv')

print('Customers:', len(customers))
print('Orders:', len(orders))
print('Customer features:', len(features))

## 1. LEFT JOIN and Row Count Validation

In [ ]:
conn = sqlite3.connect('../database/sql_joins.db')

customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
features.to_sql('customer_features', conn, if_exists='replace', index=False)

left_join = pd.read_sql_query('''
SELECT c.customer_id, c.customer_name, c.city,
       o.order_id, o.order_amount
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
''', conn)

print('Customers:', len(customers))
print('Raw LEFT JOIN rows:', len(left_join))
print('Multiplication factor:', len(left_join) / len(customers))
left_join

The LEFT JOIN produces more rows than the customer table because customer C001 has multiple orders.

## 2. Unmatched Keys

In [ ]:
no_orders = pd.read_sql_query('''
SELECT c.customer_id, c.customer_name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
''', conn)

orphan_orders = pd.read_sql_query('''
SELECT o.order_id, o.customer_id, o.order_amount
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
''', conn)

print('Customers without orders:', len(no_orders))
print('Unmatched customer percentage:', len(no_orders) / len(customers) * 100)
print('Orphaned orders:', len(orphan_orders))
print('Orphaned order percentage:', len(orphan_orders) / len(orders) * 100)

display(no_orders)
display(orphan_orders)

## 3. INNER, LEFT, and FULL OUTER JOIN

In [ ]:
inner_join = pd.read_sql_query('''
SELECT c.customer_id, o.order_id, o.order_amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
''', conn)

left_count = len(left_join)
inner_count = len(inner_join)

full_join = pd.read_sql_query('''
SELECT c.customer_id, o.order_id, o.order_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
UNION ALL
SELECT c.customer_id, o.order_id, o.order_amount
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
''', conn)

print('INNER JOIN:', inner_count)
print('LEFT JOIN:', left_count)
print('FULL OUTER JOIN:', len(full_join))

SQLite does not provide native FULL OUTER JOIN support, so the FULL OUTER result is emulated using LEFT JOIN plus UNION ALL.

## 4. Three-Table Join

In [ ]:
multi_table = pd.read_sql_query('''
SELECT c.customer_id, c.customer_name, c.city,
       o.order_id, o.order_amount,
       f.total_transactions, f.total_spent,
       f.days_since_last_purchase, f.purchase_count
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN customer_features f ON c.customer_id = f.customer_id
ORDER BY c.customer_id, o.order_id
''', conn)

print('Three-table join rows:', len(multi_table))
print('Duplicate feature keys:', features['customer_id'].duplicated().sum())
multi_table

The repository does not contain order_items or products tables. Therefore, customer_features is used as the third available customer-level table. Its customer_id key is unique, so it does not introduce unexpected row multiplication.

## 5. Validation Summary

- Customers: 5
- Orders: 4
- Raw LEFT JOIN: 6 rows
- INNER JOIN: 3 rows
- FULL OUTER JOIN: 7 rows
- Customers without orders: 3
- Orphaned orders: 1
- Three-table join: 6 rows
- All validation checks: PASS